# Chapter 33: Mapping Chemical Space

```{admonition} 🟡 Intermediate · undergraduate
:class: level-2
Assumes Chapters 21, 31 and 32.
```

"Chemical space" is the set of all possible molecules, estimated at 10⁶⁰ drug-sized compounds. We cannot list it, but we can **map** the part we have: compress fingerprints into two dimensions, cluster similar molecules, and spot gaps worth exploring.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from rdkit import Chem, DataStructs
from rdkit.Chem import rdFingerprintGenerator, Draw
from rdkit.ML.Cluster import Butina
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans

DATA_URL = "https://raw.githubusercontent.com/bahadursk/molecules-data-python/main/book/data/"
def load_data(name):
    local = Path("../data") / name
    return pd.read_csv(local if local.exists() else DATA_URL + name)

df = load_data("molecules_sim.csv").sample(1200, random_state=0).reset_index(drop=True)
mols = [Chem.MolFromSmiles(s) for s in df.smiles]
gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
fps = [gen.GetFingerprint(m) for m in mols]
X = np.array([gen.GetFingerprintAsNumPy(m) for m in mols], dtype=float)

## 33.1 PCA: the linear view

In [ ]:
pca = PCA(n_components=10).fit(X)
P = pca.transform(X)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].bar(range(1, 11), pca.explained_variance_ratio_ * 100, color="#6c4ad6")
axes[0].set(xlabel="component", ylabel="% variance", title="Fingerprint space is high-dimensional")
sc = axes[1].scatter(P[:, 0], P[:, 1], c=df.logP, cmap="viridis", s=8)
plt.colorbar(sc, ax=axes[1], label="logP"); axes[1].set(xlabel="PC1", ylabel="PC2", title="PCA map coloured by logP")
plt.tight_layout(); plt.show()

## 33.2 t-SNE: the neighbourhood view

t-SNE keeps similar molecules close together; distances between far-apart clusters are not meaningful.

In [ ]:
emb = TSNE(n_components=2, metric="jaccard", init="random", perplexity=35, random_state=0).fit_transform(X.astype(bool))
top = df.scaffold.value_counts().index[:8]
colours = dict(zip(top, plt.cm.tab10.colors))
fig, ax = plt.subplots(figsize=(8, 7))
other = ~df.scaffold.isin(top)
ax.scatter(emb[other, 0], emb[other, 1], s=6, color="lightgrey", label="other scaffolds")
for s in top:
    m = df.scaffold == s
    ax.scatter(emb[m, 0], emb[m, 1], s=12, color=colours[s], label=s[:28])
ax.legend(fontsize=7, markerscale=2, loc="best"); ax.set_title("t-SNE of Morgan fingerprints (Jaccard), by scaffold"); ax.axis("off")
plt.show()

In [ ]:
Draw.MolsToGridImage([Chem.MolFromSmiles(s) if s else Chem.MolFromSmiles("C") for s in top], legends=[f"scaffold {i+1}" for i in range(8)],
                     molsPerRow=4, subImgSize=(180, 140))

UMAP (`pip install umap-learn`) gives similar maps faster and preserves more global structure; the call is `umap.UMAP(metric="jaccard").fit_transform(X)`.

## 33.3 Property landscapes

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, col, cmap in zip(axes, ["pIC50_sim", "logS_sim", "toxic_sim"], ["plasma", "viridis", "coolwarm"]):
    sc = ax.scatter(emb[:, 0], emb[:, 1], c=df[col], s=7, cmap=cmap)
    plt.colorbar(sc, ax=ax); ax.set_title(col); ax.axis("off")
plt.tight_layout(); plt.show()

Activity forms "islands" in chemical space. The steeper the landscape (similar molecules with very different activities, called **activity cliffs**), the harder it is to model.

## 33.4 Butina clustering

Butina's algorithm {cite:p}`butina1999` groups molecules whose Tanimoto distance to a cluster centre is below a threshold.

In [ ]:
dists = []
for i in range(1, len(fps)):
    sims = DataStructs.BulkTanimotoSimilarity(fps[i], fps[:i])
    dists.extend(1 - s for s in sims)
clusters = Butina.ClusterData(dists, len(fps), 0.6, isDistData=True)
sizes = sorted((len(c) for c in clusters), reverse=True)
print(len(clusters), "clusters; largest:", sizes[:10])
centroids = [mols[c[0]] for c in sorted(clusters, key=len, reverse=True)[:8]]
Draw.MolsToGridImage(centroids, legends=[f"cluster {i+1} (n={s})" for i, s in enumerate(sizes[:8])], molsPerRow=4, subImgSize=(190, 150))

## 33.5 Diversity picking

To choose 12 molecules that cover the space for screening, use MaxMin picking:

In [ ]:
from rdkit.SimDivFilters.rdSimDivPickers import MaxMinPicker
picks = list(MaxMinPicker().LazyBitVectorPick(fps, len(fps), 12, seed=42))
fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(emb[:, 0], emb[:, 1], s=5, color="lightgrey")
ax.scatter(emb[picks, 0], emb[picks, 1], s=120, color="#e5487a", edgecolor="k", marker="*")
ax.set_title("12 diverse picks spread across the map"); ax.axis("off"); plt.show()

## 33.6 Scaffold splits: an honest test

A random split puts close analogues in both training and test sets, which flatters the model. A **scaffold split** keeps whole scaffolds out of training, mimicking the prediction of genuinely new chemistry.

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score
from sklearn.model_selection import GroupShuffleSplit, ShuffleSplit

y = df.pIC50_sim.values
groups = df.scaffold.fillna("acyclic").values
scores = {"random split": [], "scaffold split": []}
for seed in range(5):
    for name, splitter in [("random split", ShuffleSplit(1, test_size=0.2, random_state=seed)),
                           ("scaffold split", GroupShuffleSplit(1, test_size=0.2, random_state=seed))]:
        tr, te = next(splitter.split(X, y, groups))
        m = RandomForestRegressor(200, n_jobs=-1, random_state=0).fit(X[tr], y[tr])
        scores[name].append(r2_score(y[te], m.predict(X[te])))
for k, v in scores.items():
    print(f"{k:15s} R² = {np.mean(v):.2f} ± {np.std(v):.2f}")

```{admonition} 🤖 ML corner
:class: ml-corner
Always report how you split your data. Benchmark suites such as MoleculeNet {cite:p}`wu2018` recommend scaffold or time splits for exactly this reason.
```

## Further Reading

- [NumPy documentation](https://numpy.org/doc/stable/)
- [pandas user guide](https://pandas.pydata.org/docs/user_guide/)
- [Matplotlib tutorials and gallery](https://matplotlib.org/stable/tutorials/index.html)
- [The RDKit Getting Started guide](https://www.rdkit.org/docs/GettingStartedInPython.html)
- [scikit-learn user guide](https://scikit-learn.org/stable/user_guide.html)
- Works cited in this chapter are listed in full in the [References](../appendices/A5_references.md).

## Exercises

**★ 1.** Colour the PCA map by molecular weight. Which component tracks size?

**★★ 2.** Use k-means with 10 clusters on the PCA coordinates and draw one molecule from each cluster.

**★★★ 3.** Find activity cliffs: pairs with Tanimoto > 0.7 but |ΔpIC50| > 1.5. Draw the three most extreme pairs side by side.